# Chimera and multiscale synchronisation control

This notebook exercises the public `scpn_quantum_control.chimera_control` facade. It generates two finite synthetic Kuramoto–Sakaguchi regimes, measures nested order parameters, builds an analytic hierarchy target, proposes one unapplied phase step, and projects a local coupling candidate through the existing topology ledger.

The output is configuration-specific research evidence. It is not a thermodynamic-limit attractor proof, biological or EEG model, stability or controllability certificate, provider or hardware result, or deployment instruction.

In [ ]:
import numpy as np

from scpn_quantum_control.chimera_control import (
    ChimeraControlSpecification,
    HierarchyTarget,
    SyntheticChimeraConfig,
    SyntheticRegime,
    build_chimera_control_objective,
    generate_two_population_chimera,
    measure_multiscale_order_parameters,
    project_chimera_coupling,
    propose_phase_control_step,
)
from scpn_quantum_control.topology_control.constraints import (
    CouplingGraphBounds,
    TopologyConstraintLedger,
)

## 1. Generate the exact finite synthetic regimes

Both runs use the production Sakaguchi force and RK4. The reference chimera row uses strong within-population and weak between-population coupling; the synchronised control increases between-population coupling.

In [ ]:
chimera = generate_two_population_chimera(
    SyntheticChimeraConfig.for_regime(
        SyntheticRegime.CHIMERA_TRANSIENT,
        population_size=64,
    )
)
synchronised = generate_two_population_chimera(
    SyntheticChimeraConfig.for_regime(
        SyntheticRegime.SYNCHRONISED_CONTROL,
        population_size=64,
    )
)

for run in (chimera, synchronised):
    population_r = run.diagnostics.community_order_parameters
    print(
        run.config.regime.value,
        {
            "population_mean": np.mean(population_r, axis=0).tolist(),
            "chimera_index": run.diagnostics.chimera_index,
            "trajectory_digest": run.content_digest,
        },
    )

## 2. Measure the hierarchy and propose a local target step

The `population` scale has two target coherences. The `ensemble` scale has one. The objective reuses the existing analytic cluster-order gradient; the proposal performs a local backtracking step and does not actuate anything.

In [ ]:
observables = measure_multiscale_order_parameters(
    chimera.settled_phases,
    chimera.hierarchy,
)
specification = ChimeraControlSpecification(
    chimera.hierarchy,
    (
        HierarchyTarget("population", (1.0, 0.5)),
        HierarchyTarget("ensemble", (0.7,), weight=0.25),
    ),
)
objective = build_chimera_control_objective(specification)
proposal = propose_phase_control_step(objective, chimera.settled_phases[-1])

print("levels", observables.hierarchy.level_names)
print("population means", observables.level("population").mean_by_community)
print("proposal accepted", proposal.accepted)
print("objective before/after", proposal.original_value, proposal.proposed_value)

## 3. Project a local coupling candidate

The projection delegates bounds, signs, budgets, frozen edges, and hardware-edge semantics to `TopologyConstraintLedger`. A zero post-projection scalar violation for the configured constraints is not a stability or controllability result.

In [ ]:
candidate = np.array(chimera.coupling, copy=True) * 1.6
np.fill_diagonal(candidate, 0.2)
ledger = TopologyConstraintLedger(
    bounds=CouplingGraphBounds(
        0.0,
        chimera.config.intra_coupling / chimera.config.population_size,
    ),
    sign_policy="nonnegative",
    total_weight=(0.0, float(np.sum(chimera.coupling))),
)
projection = project_chimera_coupling(candidate, chimera.hierarchy, ledger)
print("violation before", projection.violations_before.total)
print("violation after", projection.violations_after.total)
print("projection digest", projection.content_digest)

## Next steps

Read the [chimera and multiscale control guide](../docs/chimera_multiscale_control.md) for shapes, errors, evidence custody, citations, and claim boundaries. Regenerate the committed evidence with `PYTHONPATH=src:oscillatools/src python scripts/run_chimera_multiscale_control_evidence.py --check`.